In [17]:
from llama_index.core.bridge.pydantic import BaseModel, Field
from llama_index.llms.ollama import Ollama
from src.classes.crossword_puzzle import CrosswordPuzzle
from src.classes.clue import Clue

In [18]:
class RankedClue(BaseModel):
    clue: str = Field(description="The clue itself, describing the word to be found.")
    number: int = Field(description="The number associated with the clue, indicating its position in the crossword grid.")
    direction: str = Field(description="The direction of the clue, either 'across' or 'down'.")
    difficulty: int = Field(description="A score from 1 to 100 indicating the difficulty of the clue, with 1 being the easiest and 100 being the hardest.")
    explanation: str = Field(description="A brief explanation of why the clue is considered to have the given difficulty score, including any wordplay, obscurity, or other factors that contribute to its difficulty.")

class Clues(BaseModel):
    clues: list[RankedClue] = Field(
        description="A list of clues ordered from easiest to hardest, based on the difficulty scores. The first clue should have the highest difficulty score, and the last clue should have the lowest difficulty score."
    )

In [19]:
llm = Ollama(
    model="llama3.1:latest",
    request_timeout=1200.0,
    context_window=1000,
    temperature=0.1,
    json_mode=True,
)

sllm = llm.as_structured_llm(Clues)

In [20]:
def generate_prompt(clues: list[Clue]) -> str:
    clues_text = "\n".join(
        f"-  ({clue.number} {clue.direction}): {clue.text} "
        for clue in clues
    )
    return f"""You are a crossword puzzle solver. You are given a list of clues for a crossword puzzle, ordered from easiest to hardest. Each clue has a difficulty score from 1 to 100, with 1 being the easiest and 100 being the hardest. For each clue, provide a brief explanation of why the clue is considered to have the given difficulty score, including any wordplay, obscurity, or other factors that contribute to its difficulty.

Clues:
{clues_text}"""

In [21]:
crossword = CrosswordPuzzle("puz_files/nytm_2025_01_01.puz")

In [22]:
prompt = generate_prompt(crossword.clues)

print(prompt)

You are a crossword puzzle solver. You are given a list of clues for a crossword puzzle, ordered from easiest to hardest. Each clue has a difficulty score from 1 to 100, with 1 being the easiest and 100 being the hardest. For each clue, provide a brief explanation of why the clue is considered to have the given difficulty score, including any wordplay, obscurity, or other factors that contribute to its difficulty.

Clues:
-  (1 across): Month that was the first of the new year in early 3-Down calendars 
-  (6 across): Love so much 
-  (7 across): Maker of Ironman Triathlon watches 
-  (8 across): Stop 
-  (9 across): Searches (for) 
-  (1 down): Soccer contest 
-  (2 down): Francophile's farewell 
-  (3 down): Like Janus, the god of beginnings 
-  (4 down): Feature of a cockatoo's head 
-  (5 down): Not-so-nice magic spells 


In [23]:
response = sllm.complete(prompt).raw
response.clues.sort(key=lambda x: x.difficulty, reverse=True)

In [24]:
response.clues

[RankedClue(clue='Not-so-nice magic spells', number=5, direction='down', difficulty=100, explanation="This clue requires some knowledge of fantasy and mythology. The answer is likely 'hexes', which refers to magical spells that are intended to harm or curse someone."),
 RankedClue(clue='Like Janus, the god of beginnings', number=3, direction='down', difficulty=90, explanation="This clue requires some knowledge of mythology and wordplay. The answer is likely 'two-faced', which refers to the fact that Janus has two faces, one looking forward and one backward."),
 RankedClue(clue="Francophile's farewell", number=2, direction='down', difficulty=80, explanation="This clue requires some knowledge of languages and cultural nuances. The answer is likely 'au revoir', which means 'until we meet again' in French."),
 RankedClue(clue="Feature of a cockatoo's head", number=4, direction='down', difficulty=70, explanation="This clue requires some knowledge of birds and their physical characteristics.